# SmartFactory AI - Etape 7 : Interprétation
Objectif : comprendre **pourquoi** le modèle prédit une panne, c'est-à-dire quelles variables comptent le plus.

Méthodes utilisées (de la plus simple à la plus fine) :
1. Importance par **impureté** (Random Forest) : rapide, mais biaisée par les variables corrélées.
2. Importance par **permutation** : on casse une variable et on mesure la perte de performance.
3. Importance par **groupes** de variables : indispensable ici car beaucoup de variables sont redondantes.
4. **Dépendance partielle** : comment la probabilité de panne évolue quand une variable change.
5. **SHAP** (optionnel) : contribution de chaque variable à chaque prédiction.
6. **Arbre de décision simple** : des règles lisibles, comparables aux règles physiques du dataset.

Prérequis : `rf_tuned.pkl` (notebook 04).

## 1. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

sns.set_theme(style="whitegrid")

BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
DATA, MODELS, FIG = BASE / "data", BASE / "models", BASE / "figures"
FIG.mkdir(exist_ok=True)
print("Projet :", BASE)

train, test = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]

rf = joblib.load(MODELS / "rf_tuned.pkl")
scaler = joblib.load(MODELS / "scaler.pkl")
feature_cols = joblib.load(MODELS / "feature_cols.pkl")
print("Variables :", feature_cols)

## 2. Importance par impureté (MDI)
Pour chaque variable, on additionne la baisse d'impureté qu'elle provoque dans tous les arbres de la forêt.

**Limite :** quand deux variables disent la même chose (Torque et Power, corrélation 0,98), l'importance est **partagée** entre elles, donc chacune paraît moins importante qu'elle ne l'est.

In [ ]:
mdi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(8, 5))
mdi.plot(kind="barh", color="tab:blue")
plt.title("Importance des variables (impureté, Random Forest optimisé)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIG / "importance_mdi.png", dpi=200, bbox_inches="tight")
plt.show()

display(mdi.sort_values(ascending=False).round(3).to_frame("Importance MDI"))

## 3. Importance par permutation (sur le test)
On mélange au hasard les valeurs d'une variable dans le test et on mesure **la baisse de PR-AUC**. Plus la baisse est grande, plus le modèle dépendait de cette variable.

C'est plus fiable que l'impureté, mais les variables corrélées restent un problème : si on mélange Torque, Power garde l'information.

In [ ]:
perm = permutation_importance(rf, X_test, y_test, scoring="average_precision",
                              n_repeats=20, random_state=42, n_jobs=-1)

perm_df = pd.DataFrame({"Permutation (baisse PR-AUC)": perm.importances_mean,
                        "Ecart-type": perm.importances_std}, index=feature_cols).sort_values("Permutation (baisse PR-AUC)")

plt.figure(figsize=(8, 5))
plt.barh(perm_df.index, perm_df["Permutation (baisse PR-AUC)"], xerr=perm_df["Ecart-type"], color="tab:orange")
plt.axvline(0, color="black", lw=0.8)
plt.title("Importance par permutation (test, PR-AUC)")
plt.xlabel("Baisse de PR-AUC quand la variable est mélangée")
plt.tight_layout()
plt.savefig(FIG / "importance_permutation.png", dpi=200, bbox_inches="tight")
plt.show()

comp = pd.concat([mdi.rename("MDI"), perm_df["Permutation (baisse PR-AUC)"]], axis=1)
comp["Rang MDI"] = comp["MDI"].rank(ascending=False).astype(int)
comp["Rang permutation"] = comp["Permutation (baisse PR-AUC)"].rank(ascending=False).astype(int)
display(comp.sort_values("Rang MDI").round(3))

## 4. Importance par groupes de variables
Les variables redondantes se regroupent naturellement (on l'a vu avec la PCA). On mélange **tout un groupe en même temps**, en conservant les relations entre ses variables : c'est la mesure la plus honnête de l'importance d'un « thème » physique.

In [ ]:
groups = {
    "Mécanique (Torque, Power, Vitesse)": ["Torque", "Power", "Rotational_speed"],
    "Usure (Tool_wear, Torque_x_Wear)": ["Tool_wear", "Torque_x_Wear"],
    "Thermique (Air, Process, Temp_diff)": ["Air_temperature", "Process_temperature", "Temp_diff"],
    "Type de produit": ["Type_enc"],
}
groups = {g: [c for c in cols if c in feature_cols] for g, cols in groups.items()}

def group_importance(model, X, y, groups, n_repeats=20, seed=42):
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, model.predict_proba(X)[:, 1])
    out = {}
    for g, cols in groups.items():
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = X[cols].values[rng.permutation(len(X))]   # lignes mélangées ensemble
            drops.append(base - average_precision_score(y, model.predict_proba(Xp)[:, 1]))
        out[g] = (np.mean(drops), np.std(drops))
    return base, out

base_ap, gi = group_importance(rf, X_test, y_test, groups)
gi_df = pd.DataFrame(gi, index=["Baisse PR-AUC", "Ecart-type"]).T.sort_values("Baisse PR-AUC")
print("PR-AUC de référence (test) :", round(base_ap, 3))

plt.figure(figsize=(9, 3.8))
plt.barh(gi_df.index, gi_df["Baisse PR-AUC"], xerr=gi_df["Ecart-type"], color="tab:green")
plt.title("Importance par groupes de variables (permutation conjointe)")
plt.xlabel("Baisse de PR-AUC")
plt.tight_layout()
plt.savefig(FIG / "importance_groupes.png", dpi=200, bbox_inches="tight")
plt.show()
display(gi_df.sort_values("Baisse PR-AUC", ascending=False).round(3))

## 5. Variables créées ou variables d'origine ?
Part de l'importance (MDI) portée par les 3 variables construites par feature engineering.

In [ ]:
created = ["Temp_diff", "Power", "Torque_x_Wear"]
share = mdi[mdi.index.isin(created)].sum() / mdi.sum() * 100
print(f"Part de l'importance portée par les variables créées : {share:.1f} %")
print(f"Part portée par les variables d'origine              : {100 - share:.1f} %")

## 6. Dépendance partielle
On fait varier **une variable** sur toute sa plage (les autres restent telles quelles) et on regarde la **probabilité moyenne de panne** prédite. Cela montre la forme de la relation : seuil, courbe en U, etc.

**Attention :** les variables créées (Power, Torque_x_Wear, Temp_diff) dépendent des autres. Faire varier Torque seul crée des combinaisons irréalistes avec Power. Lis donc ces courbes comme une **tendance**, pas comme une loi exacte.

In [ ]:
def pdp(model, X, feat, n=30):
    lo, hi = X[feat].quantile(0.01), X[feat].quantile(0.99)
    xs = np.linspace(lo, hi, n)
    ys = []
    for v in xs:
        Xt = X.copy()
        Xt[feat] = v
        ys.append(model.predict_proba(Xt)[:, 1].mean())
    return xs, np.array(ys)

sample = X_test.sample(min(1500, len(X_test)), random_state=42)
feats = ["Tool_wear", "Torque", "Rotational_speed", "Temp_diff", "Power", "Torque_x_Wear"]
feats = [f for f in feats if f in feature_cols]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), feats):
    xs, ys = pdp(rf, sample, f)
    j = feature_cols.index(f)
    ax.plot(xs * scaler.scale_[j] + scaler.mean_[j], ys, lw=2)      # retour aux unités d'origine
    ax.set_title(f)
    ax.set_xlabel(f + " (unités d'origine)")
    ax.set_ylabel("Probabilité moyenne de panne")
plt.suptitle("Dépendance partielle", y=1.01)
plt.tight_layout()
plt.savefig(FIG / "dependance_partielle.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. SHAP (optionnel)
SHAP attribue à chaque variable une **contribution** pour chaque prédiction. Il faut `pip install shap`. Si le paquet est absent, la cellule s'arrête proprement.

In [ ]:
try:
    import shap
    Xs = X_test.sample(min(800, len(X_test)), random_state=1)
    explainer = shap.TreeExplainer(rf)
    sv = explainer.shap_values(Xs)
    if isinstance(sv, list):
        sv = sv[1]
    elif getattr(sv, "ndim", 2) == 3:
        sv = sv[:, :, 1]

    shap.summary_plot(sv, Xs, show=False)
    plt.title("SHAP : contribution des variables à la probabilité de panne")
    plt.savefig(FIG / "shap_summary.png", dpi=200, bbox_inches="tight")
    plt.show()
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("Le paquet shap n'est pas installé : pip install shap")

**Lecture du graphique SHAP :** chaque point est une machine. Position horizontale : effet sur la probabilité de panne (à droite = augmente le risque). Couleur : valeur de la variable (rouge = élevée, bleu = faible). Par exemple, des points rouges à droite pour `Tool_wear` signifient « usure élevée, risque plus grand ».

In [ ]:
if SHAP_OK:
    # Explication locale d'une machine en panne bien détectée
    proba = rf.predict_proba(Xs)[:, 1]
    i = int(np.argmax(proba))
    ev = explainer.expected_value
    ev = float(ev[1]) if np.ndim(ev) > 0 else float(ev)
    orig = pd.DataFrame(scaler.inverse_transform(Xs), columns=feature_cols)

    exp = shap.Explanation(values=sv[i], base_values=ev, data=orig.iloc[i].values, feature_names=feature_cols)
    shap.plots.waterfall(exp, show=False)
    plt.title(f"Explication locale (probabilité de panne = {proba[i]:.2f})")
    plt.savefig(FIG / "shap_local.png", dpi=200, bbox_inches="tight")
    plt.show()

## 8. Un arbre simple pour lire les règles
Un arbre de **profondeur 3** entraîné sur les valeurs **d'origine** (non normalisées) donne des règles lisibles. On peut les comparer aux règles physiques du dataset (TWF, HDF, PWF, OSF).

In [ ]:
clean = pd.read_csv(DATA / "ai4i_clean.csv")
Xc, yc = clean[feature_cols], clean["Machine_failure"]
Xtr_raw, Xte_raw, ytr_raw, yte_raw = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xtr_raw, ytr_raw)
print(export_text(tree, feature_names=feature_cols))

plt.figure(figsize=(18, 8))
plot_tree(tree, feature_names=feature_cols, class_names=["Normal", "Panne"], filled=True, rounded=True, fontsize=9)
plt.title("Arbre de décision (profondeur 3, valeurs d'origine)")
plt.savefig(FIG / "arbre_regles.png", dpi=200, bbox_inches="tight")
plt.show()

from sklearn.metrics import recall_score, precision_score
p = tree.predict(Xte_raw)
print("Arbre simple (test) : recall =", round(recall_score(yte_raw, p), 3), "| precision =", round(precision_score(yte_raw, p), 3))

## 9. Cohérence avec la physique du dataset
Les modes de panne sont définis par des règles à seuil (documentation AI4I 2020). Compare avec tes graphiques :

| Mode | Règle | Variables attendues parmi les plus importantes |
|---|---|---|
| TWF | Usure entre 200 et 240 min | `Tool_wear` |
| HDF | Écart de température < 8,6 K et vitesse < 1380 rpm | `Temp_diff`, `Rotational_speed` |
| PWF | Puissance < 3500 W ou > 9000 W | `Power` (ou `Torque`) |
| OSF | Couple × usure > 11 000 / 12 000 / 13 000 selon le type | `Torque_x_Wear`, `Type_enc` |
| RNF | Hasard (0,1 %) | Aucune |

Si ton classement d'importance place ces variables en tête, le modèle a bien appris les **mécanismes physiques** et pas du bruit.

## 10. Sauvegarde

In [ ]:
out = comp.copy()
out["Groupe"] = [next((g for g, cols in groups.items() if v in cols), "") for v in out.index]
out.sort_values("MDI", ascending=False).to_csv(DATA / "importance_variables.csv")
gi_df.sort_values("Baisse PR-AUC", ascending=False).to_csv(DATA / "importance_groupes.csv")
print("Sauvegardé :", DATA / "importance_variables.csv")

## Résumé à retenir pour le rapport
1. Trois mesures d'importance sont comparées : impureté, permutation, groupes.
2. L'importance **par groupes** est la plus fiable ici, car les variables sont redondantes (Torque/Power, Tool_wear/Torque_x_Wear, Air/Process).
3. Les courbes de dépendance partielle montrent des **seuils** et des courbes en U, qui expliquent pourquoi les arbres battent les modèles linéaires.
4. SHAP donne une explication **locale** : pourquoi *cette* machine est jugée à risque.
5. Les variables importantes correspondent aux règles physiques du dataset : le modèle a appris des mécanismes réels (simulés).
6. Limite : les données sont synthétiques, donc cette cohérence est attendue.